# Partie 1 - Préparation des données

### Importation des librairies
NumPy permet la manipulation de tableaux et de matrices.

In [1]:
import numpy as nmp

### Importation de Pandas
Pandas permet de manipuler des données sous forme de tableaux (DataFrame). On configure aussi l'affichage pour voir toutes les colonnes.

In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('future.no_silent_downcasting', True)

/tmp/ipykernel_6947/1190158754.py:4: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


### Chargement du Pokédex
On charge le fichier `pokedex.csv` dans un DataFrame nommé `nosPokemons`.

In [3]:
nosPokemons = pd.read_csv("datas/pokedex.csv")

#Affichage des colonnes du Dataframe
print(nosPokemons.columns.values)

<StringArray>
[                 'NUMERO',                     'NOM',
                  'TYPE_1',                  'TYPE_2',
           'POINTS_DE_VIE',          'NIVEAU_ATTAQUE',
          'NIVEAU_DEFENSE', 'NIVEAU_ATTAQUE_SPECIALE',
 'NIVEAU_DEFENSE_SPECIALE',                 'VITESSE',
      'NOMBRE_GENERATIONS',              'LEGENDAIRE']
Length: 12, dtype: str


# Affichage des dix premières lignes de nos données

Pour afficher les dix premières lignes de nos données, nous allons utiliser la fonction Head du Dataframe nosPokemons que nous avons créé.

In [4]:
print(nosPokemons.head(10))  # Affiche les 10 premières lignes du DataFrame

   NUMERO               NOM TYPE_1  TYPE_2  POINTS_DE_VIE  NIVEAU_ATTAQUE  \
0       1        Bulbizarre  Herbe  Poison             45              49   
1       2        Herbizarre  Herbe  Poison             60              62   
2       3        Florizarre  Herbe  Poison             80              82   
3       4   Mega Florizarre  Herbe  Poison             80             100   
4       5         Salameche    Feu     NaN             39              52   
5       6         Reptincel    Feu     NaN             58              64   
6       7         Dracaufeu    Feu     Vol             78              84   
7       8  Mega Dracaufeu X    Feu  Dragon             78             130   
8       9  Mega Dracaufeu Y    Feu     Vol             78             104   
9      10          Carapuce    Eau     NaN             44              48   

   NIVEAU_DEFENSE  NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE  VITESSE  \
0              49                       65                       65     

### Transformation de la feature LEGENDAIRE
La feature `LEGENDAIRE` indique si un Pokémon est légendaire (VRAI) ou non (FAUX). Être légendaire est rare et donne plus de force, donc c'est une information utile.

Par contre, dans son état actuel, elle est lue comme une **chaîne de caractères**. Le Machine Learning ne fonctionne qu'avec des données **numériques**, donc on la transforme :
- `VRAI` → `1`
- `FAUX` → `0`

In [5]:
# remplacement de VRAI FAUX par 1 0
nosPokemonsClean = nosPokemons
nosPokemonsClean["LEGENDAIRE"] = nosPokemonsClean["LEGENDAIRE"].replace("VRAI", 1)
nosPokemonsClean["LEGENDAIRE"] = nosPokemonsClean["LEGENDAIRE"].replace("FAUX", 0)
nosPokemonsClean["LEGENDAIRE"] = nosPokemonsClean["LEGENDAIRE"].astype(int)
print(nosPokemonsClean.head())

   NUMERO              NOM TYPE_1  TYPE_2  POINTS_DE_VIE  NIVEAU_ATTAQUE  \
0       1       Bulbizarre  Herbe  Poison             45              49   
1       2       Herbizarre  Herbe  Poison             60              62   
2       3       Florizarre  Herbe  Poison             80              82   
3       4  Mega Florizarre  Herbe  Poison             80             100   
4       5        Salameche    Feu     NaN             39              52   

   NIVEAU_DEFENSE  NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE  VITESSE  \
0              49                       65                       65       45   
1              63                       80                       80       60   
2              83                      100                      100       80   
3             123                      122                      120       80   
4              43                       60                       50       65   

   NOMBRE_GENERATIONS  LEGENDAIRE  
0                   1     

### Manque-t-il des données ?
Maintenant que la feature `LEGENDAIRE` est transformée, on vérifie combien d'observations contient le jeu de données.

La propriété `shape` du DataFrame donne le nombre de **lignes** (observations, donc les Pokémon) et de **colonnes** (features).

In [6]:
#Comptage du nombre d'observations et de features
print(nosPokemonsClean.shape)

(800, 12)


Le jeu de données contient **800 observations** (Pokémon) et **12 features**, ce qui correspond aux 12 colonnes affichées plus haut.

### Y a-t-il des observations manquantes ?
La fonction `info()` donne, pour chaque feature, le nombre de valeurs **non nulles** ainsi que le **type** des données. Ça permet de repérer les valeurs manquantes.

In [7]:
#Informations sur notre jeu de données
print(nosPokemonsClean.info())

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   NUMERO                   800 non-null    int64
 1   NOM                      799 non-null    str  
 2   TYPE_1                   800 non-null    str  
 3   TYPE_2                   413 non-null    str  
 4   POINTS_DE_VIE            800 non-null    int64
 5   NIVEAU_ATTAQUE           800 non-null    int64
 6   NIVEAU_DEFENSE           800 non-null    int64
 7   NIVEAU_ATTAQUE_SPECIALE  800 non-null    int64
 8   NIVEAU_DEFENSE_SPECIALE  800 non-null    int64
 9   VITESSE                  800 non-null    int64
 10  NOMBRE_GENERATIONS       800 non-null    int64
 11  LEGENDAIRE               800 non-null    int64
dtypes: int64(9), str(3)
memory usage: 75.1 KB
None


On constate que :
- le jeu de données contient **800 observations** et **12 features**;
- il manque **1 nom** (799 valeurs sur 800);
- il manque **387 `TYPE_2`** (413 valeurs sur 800). C'est normal, car beaucoup de Pokémon n'ont qu'un seul type.

On voit aussi le type de chaque feature :
- `int64` : nombres entiers (ex. `POINTS_DE_VIE`, `VITESSE`);
- `object` : chaînes de caractères (`NOM`, `TYPE_1`, `TYPE_2`).

### À la recherche des features manquantes
Pour `TYPE_2`, les valeurs manquantes sont **normales** : un Pokémon peut ne pas avoir de type secondaire. On les laisse comme elles sont.

Pour le **nom manquant**, on ne sait pas encore si le nom d'un Pokémon a une influence sur l'issue des combats. Comme on ne peut pas l'exclure, on ne laisse pas cette valeur vide.

On commence par trouver l'observation dont la feature `NOM` est vide (`isnull()`).

In [8]:
print(nosPokemonsClean[nosPokemonsClean['NOM'].isnull()])

    NUMERO  NOM  TYPE_1 TYPE_2  POINTS_DE_VIE  NIVEAU_ATTAQUE  NIVEAU_DEFENSE  \
62      63  NaN  Combat    NaN             65             105              60   

    NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE  VITESSE  \
62                       60                       70       95   

    NOMBRE_GENERATIONS  LEGENDAIRE  
62                   1           0  


Le nom manquant se trouve à l'**observation 62** (Pokémon numéro 63, de type Combat).

Il existe deux façons de traiter une observation manquante :
- **supprimer** l'observation;
- **remplacer** la valeur par une valeur calculée ou par la vraie valeur.

Comme tous les Pokémon sont référencés dans le [Pokédex national](https://www.pokepedia.fr/Liste_des_Pok%C3%A9mon_dans_l%27ordre_du_Pok%C3%A9dex_National), on peut retrouver le vrai nom et le remplacer.

### Retrouver le nom manquant
Les Pokémon du fichier sont classés dans l'ordre du Pokédex national. Il suffit donc de regarder le Pokémon **précédent** (observation 61) et le Pokémon **suivant** (observation 63) pour retrouver le nom manquant dans le Pokédex.

In [9]:
print(nosPokemonsClean['NOM'][61])
print(nosPokemonsClean['NOM'][63])

Ferosinge
Caninos


Le Pokémon manquant se trouve entre **Férosinge** et **Caninos**. D'après le Pokédex, il s'agit de **Colossinge**.

On remplace donc la valeur manquante de l'observation 62.

In [10]:
nosPokemonsClean.loc[62, 'NOM'] = "Colossinge"

On vérifie que la modification a bien été prise en compte :

In [11]:
print(nosPokemonsClean['NOM'][62])

Colossinge


> **Remarque :** cette modification ne change que le **DataFrame** en mémoire, pas le fichier `pokedex.csv`. Pour corriger le fichier aussi, on l'ouvre dans l'éditeur et on ajoute le nom manquant :
>
> Avant : `63,,Combat,,65,105,60,60,70,95,1,FAUX`
>
> Après : `63,Colossinge,Combat,,65,105,60,60,70,95,1,FAUX`

## Place aux observations des combats
On analyse maintenant le fichier `combats.csv` avec la même démarche que pour le Pokédex :
- chargement du fichier;
- affichage des features;
- affichage des 10 premières observations;
- comptage des observations et recherche de données manquantes.

In [12]:
#Chargement des données des combats
combats = pd.read_csv("datas/combats.csv")

#Affichage des colonnes du Dataframe
print(combats.columns.values)

<StringArray>
['Premier_Pokemon', 'Second_Pokemon', 'Pokemon_Gagnant']
Length: 3, dtype: str


In [13]:
#Affichage des 10 premières lignes du Dataframe
print(combats.head(10))

   Premier_Pokemon  Second_Pokemon  Pokemon_Gagnant
0              266             298              298
1              702             701              701
2              191             668              668
3              237             683              683
4              151             231              151
5              657             752              657
6              192             134              134
7               73             545              545
8              220             763              763
9              302              31               31


In [14]:
#Comptage du nombre de lignes et de colonnes
print (combats.shape)

(50000, 3)


In [15]:
#Informations sur notre jeu de données
print (combats.info())

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Premier_Pokemon  50000 non-null  int64
 1   Second_Pokemon   50000 non-null  int64
 2   Pokemon_Gagnant  50000 non-null  int64
dtypes: int64(3)
memory usage: 1.1 MB
None


Résultat de l'analyse :
- le jeu de données contient **50 000 observations** (combats) et **3 features**;
- les features sont `Premier_Pokemon`, `Second_Pokemon` et `Pokemon_Gagnant`;
- `Pokemon_Gagnant` contient le **numéro** du Pokémon qui a gagné le combat;
- toutes les features sont de type **numérique**;
- il ne manque **aucune** donnée.

## Assemblage des observations
On a deux jeux de données : les **caractéristiques** des Pokémon et les **combats**. Il faut les réunir pour savoir, pour chaque Pokémon, combien de combats il a menés et combien il en a gagnés. Ça permettra de repérer les Pokémon qui ont plus de chances de gagner.

### Nombre de combats menés
Dans un combat, un Pokémon peut être en **première** ou en **seconde** position. La position est importante : le premier Pokémon donne le premier coup. Un Pokémon avec peu de points de vie en seconde position peut perdre avant même d'attaquer.

Exemple :

| Combat | Premier Pokémon | Second Pokémon |
|---|---|---|
| 1 | Bulbizarre | Dracaufeu |
| 2 | Dracaufeu | Colossinge |
| 3 | Dracaufeu | Pikachu |

Ce qui donne le nombre de combats par Pokémon :

| Pokémon | Nombre de combats |
|---|---|
| Bulbizarre | 1 |
| Dracaufeu | 3 |
| Colossinge | 1 |
| Pikachu | 1 |

Le nombre total de combats d'un Pokémon = nombre de fois en **première position** + nombre de fois en **seconde position**.

#### Nombre de fois en première position
`groupby()` regroupe les combats par Pokémon en première position, puis `count()` compte combien il y en a dans chaque groupe.

In [16]:
nbFoisPremierePosition = combats.groupby('Premier_Pokemon').count()
print(nbFoisPremierePosition)

                 Second_Pokemon  Pokemon_Gagnant
Premier_Pokemon                                 
1                            70               70
2                            55               55
3                            68               68
4                            62               62
5                            50               50
...                         ...              ...
796                          49               49
797                          64               64
798                          60               60
799                          75               75
800                          61               61

[784 rows x 2 columns]


Le résultat est difficile à lire au premier coup d'œil :
- l'index (première colonne) est le **numéro du Pokémon** (critère de regroupement `Premier_Pokemon`);
- les colonnes `Second_Pokemon` et `Pokemon_Gagnant` contiennent toutes les deux le **nombre de fois en première position**. Avec `count()`, chaque colonne restante reçoit le même compte.

| Numéro du Pokémon | Nb fois en 1re position | Nb fois en 1re position |
|---|---|---|
| 1 | 70 | 70 |
| 2 | 55 | 55 |
| 3 | 68 | 68 |

#### Nombre de fois en seconde position
On fait la même chose pour la seconde position.

In [17]:
nbFoisSecondePosition = combats.groupby('Second_Pokemon').count()
print(nbFoisSecondePosition)

                Premier_Pokemon  Pokemon_Gagnant
Second_Pokemon                                  
1                            63               63
2                            66               66
3                            64               64
4                            63               63
5                            62               62
...                         ...              ...
796                          56               56
797                          67               67
798                          59               59
799                          69               69
800                          60               60

[784 rows x 2 columns]


#### Nombre total de combats
On additionne les deux résultats pour obtenir le nombre total de combats par Pokémon.

In [18]:
nombreTotalDeCombats = nbFoisPremierePosition + nbFoisSecondePosition
print(nombreTotalDeCombats)

                 Pokemon_Gagnant  Premier_Pokemon  Second_Pokemon
Premier_Pokemon                                                  
1                            133              NaN             NaN
2                            121              NaN             NaN
3                            132              NaN             NaN
4                            125              NaN             NaN
5                            112              NaN             NaN
...                          ...              ...             ...
796                          105              NaN             NaN
797                          131              NaN             NaN
798                          119              NaN             NaN
799                          144              NaN             NaN
800                          121              NaN             NaN

[784 rows x 3 columns]


Encore une fois, il ne faut pas se fier aux noms de colonnes : l'index est le **numéro du Pokémon** et les valeurs sont le **nombre total de combats**.

Pandas fait l'addition des deux tableaux avec un simple `+` en alignant automatiquement les numéros de Pokémon. Sans Pandas, il aurait fallu parcourir les deux tableaux nous-mêmes pour additionner les valeurs Pokémon par Pokémon.

### Nombre de combats gagnés
On calcule maintenant le nombre de victoires de chaque Pokémon. Ça permettra aussi de repérer les Pokémon qui n'ont **jamais gagné**, et qu'il vaudrait mieux éviter d'utiliser en combat.

In [19]:
nombreDeVictoires = combats.groupby('Pokemon_Gagnant').count()
print(nombreDeVictoires)

                 Premier_Pokemon  Second_Pokemon
Pokemon_Gagnant                                 
1                             37              37
2                             46              46
3                             89              89
4                             70              70
5                             55              55
...                          ...             ...
796                           39              39
797                          116             116
798                           60              60
799                           89              89
800                           75              75

[783 rows x 2 columns]


#### Lecture du résultat
Le tableau se lit ainsi :
- **l'index** (colonne de gauche) est le **numéro du Pokémon**;
- **les deux colonnes** (`Premier_Pokemon` et `Second_Pokemon`) contiennent la même valeur : le **nombre de combats gagnés** par ce Pokémon.

Par exemple, la ligne `1   37   37` veut dire que le Pokémon n°1 a gagné **37 combats**.

Comme le tableau est long, pandas n'affiche que les 5 premières et les 5 dernières lignes. Les `...` cachent les autres. Pour voir un Pokémon en particulier, on utilise `.loc[numéro]`.

In [20]:
# Exemple : le Pokémon n°228
print("Victoires :", nombreDeVictoires.loc[228, 'Premier_Pokemon'])
print("Combats   :", nombreTotalDeCombats.loc[228, 'Pokemon_Gagnant'])

Victoires : 104
Combats   : 150


L'index est le **numéro du Pokémon** et les valeurs sont son **nombre de victoires**.

| Pokémon | Nombre de combats gagnés |
|---|---|
| 1 | 37 |
| 2 | 46 |
| 3 | 89 |
| 4 | 70 |
| … | … |
| 228 | 104 |

Par exemple, en comparant avec le nombre total de combats, on voit que le Pokémon 228 a gagné 104 combats sur 150.

In [21]:
nombreDeVictoires.info()

<class 'pandas.DataFrame'>
Index: 783 entries, 1 to 800
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Premier_Pokemon  783 non-null    int64
 1   Second_Pokemon   783 non-null    int64
dtypes: int64(2)
memory usage: 34.5 KB


Le résultat ne contient que **783 observations** alors qu'il y a 800 Pokémon. Ça veut dire que **certains Pokémon n'ont jamais gagné** de combat : ils n'apparaissent jamais dans la colonne `Pokemon_Gagnant`.

## Agrégation des données avec le Pokédex
On va maintenant réunir les statistiques de combat avec le Pokédex pour avoir **toutes les données au même endroit**.

On crée d'abord une liste qui contient, pour chaque Pokémon :
- son **numéro**;
- le **nombre de combats** menés;
- le **nombre de combats gagnés**;
- le **pourcentage de victoires** (victoires ÷ combats).

In [22]:
listeAAgreger = combats.groupby('Pokemon_Gagnant').count()
listeAAgreger.sort_index()

#On ajoute le nombre de combats
listeAAgreger['NBR_COMBATS'] = nbFoisPremierePosition.Pokemon_Gagnant + nbFoisSecondePosition.Pokemon_Gagnant

#On ajoute le nombre de victoires
listeAAgreger['NBR_VICTOIRES'] = nombreDeVictoires.Premier_Pokemon

#On calcule le pourcentage de victoires
listeAAgreger['POURCENTAGE_DE_VICTOIRES'] = nombreDeVictoires.Premier_Pokemon / (nbFoisPremierePosition.Pokemon_Gagnant + nbFoisSecondePosition.Pokemon_Gagnant)

#On affiche la nouvelle liste
print(listeAAgreger)

                 Premier_Pokemon  Second_Pokemon  NBR_COMBATS  NBR_VICTOIRES  \
Pokemon_Gagnant                                                                
1                             37              37          133             37   
2                             46              46          121             46   
3                             89              89          132             89   
4                             70              70          125             70   
5                             55              55          112             55   
...                          ...             ...          ...            ...   
796                           39              39          105             39   
797                          116             116          131            116   
798                           60              60          119             60   
799                           89              89          144             89   
800                           75        

#### Explication du code
- **`groupby('Pokemon_Gagnant').count()`** sert surtout à obtenir la **liste des numéros de Pokémon** en index. On aurait aussi pu partir de `nbFoisPremierePosition` ou `nbFoisSecondePosition`. Ce qui compte, c'est d'avoir les numéros, qui serviront de **clé** pour l'agrégation avec le Pokédex.
- **`sort_index()`** trie les numéros en ordre croissant, comme dans le Pokédex.
- On ajoute ensuite 3 nouvelles colonnes : `NBR_COMBATS`, `NBR_VICTOIRES` et `POURCENTAGE_DE_VICTOIRES`.

#### Lecture du résultat

| Pokémon | NBR_COMBATS | NBR_VICTOIRES | POURCENTAGE_DE_VICTOIRES |
|---|---|---|---|
| 1 | 133 | 37 | 0.278 |
| 2 | 121 | 46 | 0.380 |
| 3 | 132 | 89 | 0.674 |
| 4 | 125 | 70 | 0.560 |

Par exemple, le Pokémon n°3 a gagné **89 combats sur 132**, soit environ **67 %** de victoires.

Les colonnes `Premier_Pokemon` et `Second_Pokemon` sont des restes du `count()` : on peut les ignorer.

### Fusion avec le Pokédex
Il ne reste plus qu'à réunir la liste `listeAAgreger` avec le DataFrame `nosPokemonsClean` (le Pokédex). On utilise la fonction `merge()` :
- **`left_on='NUMERO'`** : la clé côté Pokédex est la feature `NUMERO`;
- **`right_index=True`** : la clé côté `listeAAgreger` est son **index** (le numéro du Pokémon);
- **`how='left'`** : on garde **les 800 Pokémon** du Pokédex, même ceux qui n'ont jamais gagné.

In [23]:
nouveauPokedex = nosPokemonsClean.merge(listeAAgreger, left_on='NUMERO', right_index = True, how='left')
print(nouveauPokedex.describe())

         NUMERO  POINTS_DE_VIE  NIVEAU_ATTAQUE  NIVEAU_DEFENSE  \
count  800.0000     800.000000      800.000000      800.000000   
mean   400.5000      69.258750       79.001250       73.842500   
std    231.0844      25.534669       32.457366       31.183501   
min      1.0000       1.000000        5.000000        5.000000   
25%    200.7500      50.000000       55.000000       50.000000   
50%    400.5000      65.000000       75.000000       70.000000   
75%    600.2500      80.000000      100.000000       90.000000   
max    800.0000     255.000000      190.000000      230.000000   

       NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE     VITESSE  \
count               800.000000               800.000000  800.000000   
mean                 72.820000                71.902500   68.277500   
std                  32.722294                27.828916   29.060474   
min                  10.000000                20.000000    5.000000   
25%                  49.750000                50.0

# Partie 2 – Analyse des données

Dans la partie 1, on a construit un jeu d'observations (`nouveauPokedex`). On va maintenant l'analyser pour trouver quelles features aident à **gagner un combat**.

## Un peu de statistiques
La fonction `describe()` de Pandas donne un résumé statistique de chaque feature numérique.

In [24]:
print(nouveauPokedex.describe())

         NUMERO  POINTS_DE_VIE  NIVEAU_ATTAQUE  NIVEAU_DEFENSE  \
count  800.0000     800.000000      800.000000      800.000000   
mean   400.5000      69.258750       79.001250       73.842500   
std    231.0844      25.534669       32.457366       31.183501   
min      1.0000       1.000000        5.000000        5.000000   
25%    200.7500      50.000000       55.000000       50.000000   
50%    400.5000      65.000000       75.000000       70.000000   
75%    600.2500      80.000000      100.000000       90.000000   
max    800.0000     255.000000      190.000000      230.000000   

       NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE     VITESSE  \
count               800.000000               800.000000  800.000000   
mean                 72.820000                71.902500   68.277500   
std                  32.722294                27.828916   29.060474   
min                  10.000000                20.000000    5.000000   
25%                  49.750000                50.0

### Exemple : la feature `NIVEAU_ATTAQUE`

| Statistique | Définition | Valeur |
|---|---|---|
| count | Nombre d'observations | 800 |
| mean | Moyenne | 79 |
| std | Écart type | 32 |
| min | Valeur minimale | 5 |
| 25 % | 1er quartile | 55 |
| 50 % | Médiane | 75 |
| 75 % | 3e quartile | 100 |
| max | Valeur maximale | 190 |

- **count** : 800 valeurs, donc aucune donnée manquante.
- **mean** : l'attaque moyenne est de 79. En dessous, le niveau est plutôt faible, au-dessus plutôt fort.
- **std** : l'écart type mesure à quel point les valeurs s'éloignent de la moyenne. Ici, 32 est élevé : les niveaux d'attaque sont très variés.
- **min / max** : certains Pokémon ont seulement 5 d'attaque, d'autres jusqu'à 190.
- **Quartiles** : 25 % des Pokémon ont moins de 55, 50 % moins de 75 et 75 % moins de 100. Donc seulement **25 %** dépassent 100 d'attaque. Pour gagner, il vaut mieux choisir un Pokémon de ce dernier quartile.

### Analyse globale
- Les combats dépendent des données **numériques** (vitesse, attaque, etc.). On peut donc exclure `NOM`, `TYPE_1` et `TYPE_2` de cette analyse.
- `NUMERO` est seulement un **identifiant**, il n'a aucun lien avec la victoire.
- Certains Pokémon n'ont qu'**1 point de vie** : il faut éviter de les utiliser.
- 25 % des Pokémon ont une **vitesse supérieure à 90**. Être rapide peut être un avantage.
- `NBR_COMBATS` a un écart type d'environ **11**, le plus bas de toutes les features. Tous les Pokémon ont donc combattu à peu près le même nombre de fois (environ 127). Le jeu de données est **équilibré** : aucun Pokémon n'est surreprésenté, ce qui donne une image fidèle de la réalité.